# 03 — Mining & Clustering

Khai phá tri thức từ dữ liệu doanh số siêu thị:
- Luật kết hợp (Apriori) — giỏ hàng theo hoá đơn, gợi ý combo/cross-sell
- Phân cụm khách hàng (KMeans + HAC) — đặc trưng RFM, hồ sơ cụm

In [ ]:
# %% [import] Thư viện và cấu hình
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.data import load_params, load_processed_data
from src.features import build_customer_features, scale_features
from src.mining import run_association_pipeline, format_rules, get_top_rules
from src.mining import run_clustering_pipeline, elbow_method, reduce_pca
from src.mining.clustering import run_hac, evaluate_k
from src.evaluation import save_table, summarize_clusters, summarize_association_rules
from src.visualization import plot_elbow, plot_clusters_pca

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
# %% [load] Đọc dữ liệu đã tiền xử lý
params = load_params()
df     = load_processed_data(params)
print(f"Shape: {df.shape}")
df.head()

## PHẦN A — Luật kết hợp (Association Rules)

### 1. Xây dựng giỏ hàng (Market Basket)

In [ ]:
# %% [basket] Chuẩn hoá giỏ hàng: mỗi Order ID là 1 giao dịch, item là Sub-Category
from src.mining.association import build_basket

basket = build_basket(df, params)
print(f"Basket shape : {basket.shape}")
print(f"Items        : {list(basket.columns)}")
basket.head()

### 2. Chạy Apriori — Frequent Itemsets

In [ ]:
# %% [apriori] Chạy Apriori với min_support từ params (0.02)
from src.mining.association import run_apriori

frequent_itemsets = run_apriori(basket, params)
print(f"Frequent itemsets: {len(frequent_itemsets)}")
print(f"\nTop 10 theo support:")
frequent_itemsets.head(10)

In [ ]:
# %% [plot] Phân phối support theo độ dài itemset
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(frequent_itemsets["support"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Phân phối Support")
axes[0].set_xlabel("Support")

length_counts = frequent_itemsets["length"].value_counts().sort_index()
sns.barplot(x=length_counts.index, y=length_counts.values, ax=axes[1], palette="Blues_r")
axes[1].set_title("Số itemsets theo độ dài")
axes[1].set_xlabel("Độ dài itemset")
axes[1].set_ylabel("Số lượng")

plt.tight_layout()
plt.savefig("../outputs/figures/frequent_itemsets.png", dpi=150, bbox_inches="tight")
plt.show()

### 3. Sinh luật kết hợp

In [ ]:
# %% [rules] Sinh luật kết hợp lọc theo min_confidence=0.3 và min_lift=1.0
from src.mining.association import run_association_rules

rules = run_association_rules(frequent_itemsets, params)
print(f"Tổng số luật: {len(rules)}")
print(f"\nPhân phối lift:")
rules["lift"].describe()

In [ ]:
# %% [plot] Scatter plot confidence vs lift, kích thước = support
fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(
    rules["confidence"], rules["lift"],
    s=rules["support"] * 1000,
    alpha=0.6, c=rules["lift"], cmap="YlOrRd",
)
plt.colorbar(scatter, ax=ax, label="Lift")
ax.set_title("Confidence vs Lift (kích thước = Support)")
ax.set_xlabel("Confidence")
ax.set_ylabel("Lift")
plt.tight_layout()
plt.savefig("../outputs/figures/rules_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% [result] Top 10 luật kết hợp mạnh nhất theo lift
top_rules = format_rules(get_top_rules(rules, n=10))
save_table(top_rules, "top_association_rules.csv", params)
top_rules

In [ ]:
# %% [crosssell] Lọc luật cross-sell giữa Furniture và Technology
from src.mining.association import filter_crosssell_rules

crosssell = format_rules(filter_crosssell_rules(rules, "Chairs", "Phones"))
print(f"Cross-sell rules (Chairs → Phones): {len(crosssell)}")
crosssell.head()

In [ ]:
# %% [save] Lưu toàn bộ frequent itemsets và rules ra outputs/tables/
save_table(frequent_itemsets.head(50), "frequent_itemsets.csv", params)
save_table(format_rules(rules),        "all_association_rules.csv", params)
print("Đã lưu kết quả luật kết hợp.")

## PHẦN B — Phân cụm khách hàng (Clustering)

### 4. Chuẩn bị đặc trưng RFM

In [ ]:
# %% [feature] Xây dựng và chuẩn hoá đặc trưng khách hàng (RFM + AOV + Diversity)
customer_features       = build_customer_features(df, params)
customer_scaled, scaler = scale_features(customer_features)

feature_cols = [c for c in customer_scaled.columns if c != "Customer ID"]
X = customer_scaled[feature_cols].values

print(f"Customer features shape: {customer_features.shape}")
customer_features.describe()

### 5. Tìm số cụm tối ưu (Elbow + Silhouette)

In [ ]:
# %% [elbow] Elbow Method — tìm k tối ưu theo inertia
inertias = elbow_method(X)
plot_elbow(inertias, params)
plt.show()

In [ ]:
# %% [evaluate] Silhouette Score và Davies-Bouldin Index theo từng k
eval_df = evaluate_k(X)
print(eval_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=eval_df, x="k", y="silhouette",     ax=axes[0], marker="o", color="steelblue")
axes[0].set_title("Silhouette Score theo k")
sns.lineplot(data=eval_df, x="k", y="davies_bouldin", ax=axes[1], marker="o", color="coral")
axes[1].set_title("Davies-Bouldin Index theo k")
plt.tight_layout()
plt.savefig("../outputs/figures/clustering_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Chạy KMeans và HAC

In [ ]:
# %% [kmeans] Chạy KMeans với n_clusters=4 từ params
df_clustered, profile, km_model = run_clustering_pipeline(customer_scaled, params)
print(f"\nCluster distribution:\n{df_clustered['Cluster'].value_counts().sort_index()}")

In [ ]:
# %% [hac] Chạy HAC (Agglomerative) để so sánh với KMeans
from src.mining.clustering import run_hac, assign_clusters

hac_labels, hac_model = run_hac(X, params)
df_hac = assign_clusters(customer_scaled, hac_labels, col="Cluster_HAC")
print(f"\nHAC Cluster distribution:\n{df_hac['Cluster_HAC'].value_counts().sort_index()}")

### 7. Visualize và hồ sơ cụm

In [ ]:
# %% [plot] PCA 2D scatter plot các cụm KMeans
X_2d = reduce_pca(X)
plot_clusters_pca(X_2d, df_clustered["Cluster"].values, params)
plt.show()

In [ ]:
# %% [plot] Heatmap hồ sơ cụm theo đặc trưng RFM
profile_display = profile.set_index("Cluster")[feature_cols]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(profile_display.T, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax)
ax.set_title("Hồ sơ cụm — Giá trị trung bình các đặc trưng")
plt.tight_layout()
plt.savefig("../outputs/figures/cluster_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% [profile] In hồ sơ cụm đầy đủ và lưu CSV
print("=== HỒ SƠ CỤM ===")
print(profile.to_string(index=False))
save_table(profile, "cluster_profile.csv", params)